# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 3
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 3


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
adata = sc.read(f'./{seed}/embryo_11.5_S1_S4_Splane.h5ad')

adata[0].obs

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,annotation,Regulon - Alx3,Regulon - Arx,Regulon - Barhl2,Regulon - Bclaf1,Regulon - Bhlhe22,...,Meninges,Mesenchyme,Pancreas primordium,Sclerotome,Spinal cord,Surface ectoderm,Sympathetic nerve,Urogenital ridge,slice_id,spatial_domain
cell_name,,,,,,,,,,,,,,,,,,,,,
144_156,311,5.743003,542,6.297109,Cavity,0.052135,0.011056,0.040633,0.039881,0.032986,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,12


In [5]:
slices = [0,1,2,3]

adata_list=[]

for i in range(4):
    adata_list.append(adata[adata.obs['slice_id'] == slices[i]])
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

In [6]:
adata_list[1]

AnnData object with n_obs × n_vars = 30756 × 24088
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - Alx3', 'Regulon - Arx', 'Regulon - Barhl2', 'Regulon - Bclaf1', 'Regulon - Bhlhe22', 'Regulon - Bmyc', 'Regulon - Brf1', 'Regulon - Cdx2', 'Regulon - Cdx4', 'Regulon - Cebpa', 'Regulon - Cebpb', 'Regulon - Cebpe', 'Regulon - Cebpg', 'Regulon - Clock', 'Regulon - Creb1', 'Regulon - Creb3l2', 'Regulon - Ctcf', 'Regulon - Cux1', 'Regulon - Dbx1', 'Regulon - Dlx1', 'Regulon - Dlx2', 'Regulon - Dlx5', 'Regulon - Dlx6', 'Regulon - E2f3', 'Regulon - E2f4', 'Regulon - E2f6', 'Regulon - Egr4', 'Regulon - Elf2', 'Regulon - Elk3', 'Regulon - Elk4', 'Regulon - Emx1', 'Regulon - Emx2', 'Regulon - Esrrg', 'Regulon - Ets1', 'Regulon - Evx1', 'Regulon - Evx2', 'Regulon - Ezh2', 'Regulon - Fli1', 'Regulon - Foxa1', 'Regulon - Foxa2', 'Regulon - Foxa3', 'Regulon - Foxc1', 'Regulon - Foxc2', 'Regulon - Foxf2', 'Regulon - Foxp1', 'Regulo

Plot all raw slices stacked in a single figure. Cells colored by spatial domain.First, we define the color of each spatial domain.

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Alignment slice 2 to 1
Alignment slice 3 to 2
Runtime: 859.0820951461792 s


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

## LAA & ARI

In [8]:
from sklearn.metrics import pairwise
from sklearn.metrics import adjusted_rand_score

def cal_layer_based_alignment_result_mHypothalamus(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['annotation'])
    labels.extend(s2.obs['annotation'])

    res = []
    #l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    ir = 0
    max_smallerThan0 = []
    sum_smallerThan0 = []
    min_smallerThan0 = []

    for i, elem in enumerate(alignment):
        #if np.sum(elem)>0:
        if np.max(elem) - np.min(elem) != 0:
            if labels[i] == labels[elem.argmax() + alignment.shape[0]]:
                cnt0 += 1
        else:
            ir += 1
            max_smallerThan0.append(np.max(elem))
            sum_smallerThan0.append(np.sum(elem))
            min_smallerThan0.append(np.min(elem))
    print("the number of sum alignment equals 0:", str(ir))
    if len(sum_smallerThan0)>0:
        print("sum in sum alignment equals 0 is between", np.min(sum_smallerThan0), np.max(sum_smallerThan0))
        print("max in sum alignment equals 0 is between", np.min(max_smallerThan0), np.max(max_smallerThan0))
        print("min in sum alignment equals 0 is between", np.min(min_smallerThan0), np.max(min_smallerThan0))

    res.extend([cnt0])
    return res


def compute_alignment_ari(pi, sliceA, sliceB):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['annotation'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['annotation']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['annotation'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['annotation'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

In [ ]:
slices = [0,1,2,3]

ari_values = []
laa_values = []

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    alignment = pairwise.euclidean_distances(f1, f2)
    alignment = np.exp(-alignment)
    
    # name1 = slices[target_ind].split('.')[-1]
    # name2 = slices[source_ind].split('.')[-1]
    # Save as .pkl
#     alignment_df = pd.DataFrame(alignment)
#     alignment_df.to_pickle(f'./{seed}/{name1}_{name2}_AlignmentPi.pkl')
    
    res = cal_layer_based_alignment_result_mHypothalamus(alignment, adata_list[source_ind], adata_list[target_ind])
    laa = res[0]/len(adata_list[source_ind].obs['annotation'])
    ari = compute_alignment_ari(alignment, adata_list[target_ind], adata_list[source_ind])
    ari_values.append(ari)
    laa_values.append(laa)
    print(laa, ari)

# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame([laa_values], columns=header_row)

the number of sum alignment equals 0: 0
0.1066458577188191 0.5027445982541942
the number of sum alignment equals 0: 0
0.09298852668002186 0.26062629069631493
the number of sum alignment equals 0: 0
0.08591389114541023 0.35628786339686985


In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/embryo_E11.5_S1_S4_ARI.csv')
laa_df.to_csv(f'./{seed}/embryo_E11.5_S1_S4_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [11]:
c_dict = {'Blood vessel': '#1f77b4',
 'Brain': '#aec7e8',
 'Branchial arch': '#ff7f0e',
 'Cartilage primordium': '#ffbb78',
 'Cavity': '#2ca02c',
 'Choroid plexus': '#98df8a',
 'Connective tissue': '#d62728',
 'Dermomyotome': '#ff9896',
 'Dorsal root ganglion': '#9467bd',
 'Facial nerve': '#c5b0d5',
 'GI tract': '#8c564b',
 'Head mesenchyme': '#c49c94',
 'Heart': '#e377c2',
 'Inner ear': '#f7b6d2',
 'Jaw and tooth': '#7f7f7f',
 'Liver': '#c7c7c7',
 'Lung primordium': '#bcbd22',
 'Meninges': '#dbdb8d',
 'Mesenchyme': '#17becf',
 'Pancreas primordium': '#9edae5',
 'Sclerotome': '#393b79',
 'Spinal cord': '#5254a3',
 'Surface ectoderm': '#6b6ecf',
 'Sympathetic nerve': '#9c9ede',
 'Urogenital ridge': '#637939'}

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = adata_list[i].obs['slice_id'] + 1
    loc['original_clusters'] = adata_list[i].obs['annotation']
    loc['celltype_colors'] = adata_list[i].obs['annotation'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_embryo_E11.5_S1_S4_aligned_coordinates_colors.csv')

In [13]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
cell_name,,,,,,
144_156,51.291039,-97.052612,0,1,Cavity,#2ca02c
144_157,51.633057,-96.112930,0,1,Meninges,#dbdb8d
144_158,51.975075,-95.173233,0,1,Meninges,#dbdb8d
144_159,52.317101,-94.233536,0,1,Meninges,#dbdb8d
144_160,52.659119,-93.293854,0,1,Meninges,#dbdb8d
...,...,...,...,...,...,...
99_95,46.818890,-72.415298,3,4,Cavity,#2ca02c
99_96,47.079933,-71.449966,3,4,Cavity,#2ca02c
99_97,47.340984,-70.484634,3,4,Head mesenchyme,#c49c94


In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
seed = 14
df = pd.read_csv(f'./{seed}/SPACEL_embryo_E11.5_S1_S4_aligned_coordinates_colors.csv')
coords = df[['X', 'Y', 'Z']]
# coords['Z'] = coords['Z']

In [ ]:
fig = plot_3d(coords.values, val=None, color=df['celltype_colors'], elev=230, azim=210, roll=180, frameon=False, show=True, s=15, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_lymph_3dviz_angle230_210_180_s=15.pdf', bbox_inches="tight")